# Fixed-policy Fisher eigenspectrum

This notebook inspects the undamped empirical Fisher artifacts produced by `python -m fisher_analysis.run_fisher_analysis`. Set `FISHER_RESULTS_DIR` to load a non-default run.

In [ ]:
import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

override = os.environ.get("FISHER_RESULTS_DIR")
if override:
    results_dir = Path(override).expanduser().resolve()
else:
    candidates = [
        Path("fisher_analysis/results/hopper_width_sweep"),
        Path("results/hopper_width_sweep"),
    ]
    results_dir = next((path.resolve() for path in candidates if path.exists()), candidates[0].resolve())

if not results_dir.exists():
    raise FileNotFoundError(f"No Fisher results found at {results_dir}")
print(f"Loading {results_dir}")

In [ ]:
with (results_dir / "config.json").open(encoding="utf-8") as handle:
    config = json.load(handle)
with (results_dir / "summary.csv").open(newline="", encoding="utf-8") as handle:
    summary = list(csv.DictReader(handle))

spectra = {}
for width in config["widths"]:
    with np.load(results_dir / f"fisher_width_{width}.npz") as archive:
        spectra[width] = {name: archive[name].copy() for name in archive.files}

columns = ["width", "matrix_dimension", "sample_count", "trace", "numerical_rank", "effective_rank", "stable_rank", "positive_condition_number", "components_90", "components_95", "components_99"]
print(" | ".join(columns))
print(" | ".join(["---"] * len(columns)))
for row in summary:
    print(" | ".join(row[column] for column in columns))

In [ ]:
dimensions = {width: spectra[width]["fisher"].shape[0] for width in config["widths"]}
sample_counts = {width: int(spectra[width]["total_sample_count"]) for width in config["widths"]}
print("Matrix dimensions:", dimensions)
print("Pooled samples:", sample_counts)
for width, data in spectra.items():
    np.testing.assert_allclose(data["fisher"], data["fisher"].T, rtol=0.0, atol=float(data["psd_tolerance"]))
    np.testing.assert_allclose(np.trace(data["fisher"]), data["eigenvalues"].sum(), rtol=1e-10)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.5))
for width, data in spectra.items():
    values = data["eigenvalues"]
    floor = max(float(data["rank_tolerance"]), np.finfo(np.float64).tiny)
    ax.semilogy(np.arange(1, values.size + 1), np.maximum(values, floor), label=f"width {width} (P={values.size})")
ax.set(xlabel="Principal-component index", ylabel="Eigenvalue", title="Undamped empirical Fisher eigenspectrum")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.5))
for width, data in spectra.items():
    values = data["trace_normalized_eigenvalues"]
    trace = float(np.trace(data["fisher"]))
    floor = float(data["rank_tolerance"]) / trace if trace > 0 else np.finfo(np.float64).tiny
    ax.semilogy(np.arange(1, values.size + 1), np.maximum(values, floor), label=f"width {width}")
ax.set(xlabel="Principal-component index", ylabel="Eigenvalue / Fisher trace", title="Trace-normalized Fisher eigenspectrum")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.5))
for width, data in spectra.items():
    cumulative = data["cumulative_explained_trace"]
    ax.plot(np.arange(1, cumulative.size + 1), cumulative, label=f"width {width}")
for level in (0.90, 0.95, 0.99):
    ax.axhline(level, color="#6B7280", linewidth=0.8, linestyle="--")
ax.set(xlabel="Number of principal components", ylabel="Cumulative explained Fisher trace", title="Cumulative Fisher trace", ylim=(0, 1.01))
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Optional NPG damping view

NPG replaces the undamped eigenvalues $\lambda_i$ with $\lambda_i + \delta$. Set `NPG_DAMPING` to inspect a different diagonal shift, or set `SHOW_NPG_DAMPING=0` to skip the plot.

In [ ]:
show_damping = os.environ.get("SHOW_NPG_DAMPING", "1") != "0"
damping = float(os.environ.get("NPG_DAMPING", "0.01"))
if damping < 0:
    raise ValueError("NPG_DAMPING must be non-negative")

if show_damping:
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for width, data in spectra.items():
        original = data["eigenvalues"]
        shifted = original + damping
        positive_condition = shifted[0] / shifted[-1] if shifted[-1] > 0 else np.inf
        print(f"width {width}: damped condition number = {positive_condition:.6g}")
        ax.semilogy(np.arange(1, shifted.size + 1), shifted, label=f"width {width}")
    ax.set(xlabel="Principal-component index", ylabel=f"Eigenvalue + {damping:g}", title="Fisher spectrum after NPG diagonal damping")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Damping view disabled")